Import e Caricamento grafi salvati 

In [1]:
import sys
if 'utils' in sys.modules:
    del sys.modules['utils']
sys.path.append('../src')

import torch
import networkx as nx
import numpy as np
import matplotlib.pyplot as plt
from utils import load_dataset, EDGE_FEATURES

# Carica un grafo di esempio (CONFIG_D, primo grafo di train)
sample = torch.load('../outputs/graphs/CONFIG_D/train/graph_000.pt', weights_only=False)
print(sample)

HeteroData(
  node={
    x=[4001, 3],
    num_nodes=4001,
  },
  (node, comm, node)={
    edge_index=[2, 4146],
    edge_attr=[4146, 8],
    edge_label=[4146],
  },
  (node, context, node)={
    edge_index=[2, 4146],
    edge_attr=[4146, 4],
    edge_label=[4146],
  },
  (node, knowledge, node)={
    edge_index=[2, 4146],
    edge_attr=[4146, 8],
    edge_label=[4146],
  }
)


In [2]:
import pandas as pd

df = load_dataset('../data/ton_iot_20pct.csv')
df['window'] = df['Timestamp'].dt.floor(f'{WINDOW_HOURS}h')

window_counts = df.groupby('window').size()
valid_windows = window_counts[window_counts >= 100].index

# Prima finestra valida
first_window = valid_windows[0]
window_df = df[df['window'] == first_window].copy()
G = build_graph(window_df, edge_types=['comm'])

print(f"Finestra: {first_window}")
print(f"Nodi: {G.number_of_nodes()}")
print(f"Archi: {G.number_of_edges()}")

NameError: name 'WINDOW_HOURS' is not defined

In [ ]:
community_map = run_lpa(G)
n_communities = len(set(community_map.values()))
print(f"Community trovate: {n_communities}")

sizes = Counter(community_map.values())
size_dist = Counter(sizes.values())
print(f"\nDistribuzione dimensione community:")
for size, count in sorted(size_dist.items()):
    print(f"  Community da {size} nodi: {count}")

In [ ]:
community_stats = []

for window in valid_windows:
    window_df = df[df['window'] == window].copy()
    G = build_graph(window_df, edge_types=['comm'])
    community_map = run_lpa(G)
    n_comm = len(set(community_map.values()))
    community_stats.append({
        'window': window,
        'n_nodes': G.number_of_nodes(),
        'n_edges': G.number_of_edges(),
        'n_communities': n_comm
    })

stats_df = pd.DataFrame(community_stats)
print(stats_df.describe().round(1))

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

axes[0].plot(stats_df['window'], stats_df['n_nodes'],
             color='#3498db', linewidth=2, label='Nodi')
axes[0].plot(stats_df['window'], stats_df['n_edges'],
             color='#e74c3c', linewidth=2, label='Archi')
axes[0].set_title('Nodi e Archi per Finestra Temporale', fontweight='bold')
axes[0].legend()
axes[0].tick_params(axis='x', rotation=45)

axes[1].plot(stats_df['window'], stats_df['n_communities'],
             color='#2ecc71', linewidth=2)
axes[1].set_title('Community Rilevate per Finestra Temporale', fontweight='bold')
axes[1].set_ylabel('N. Community')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('../outputs/figures/community_over_time.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figura salvata.")